In [1]:
!pip install -q python-dotenv

In [2]:
import os
from dataclasses import dataclass

In [3]:
# Token prices in USD per million tokens

INPUT_PRICE_PER_MILLION = 0.30
OUTPUT_PRICE_PER_MILLION = 1.20

# Usage assumptions

REQUESTS_PER_DAY = 2000

AVG_INPUT_TOKENS = 1200
AVG_OUTPUT_TOKENS = 300

PEAK_REQUESTS_PER_MINUTE = 30

# Rate limits

RPM_LIMIT = 60
TPM_LIMIT = 100000

# Budget

MONTHLY_BUDGET_USD = 50
DAILY_BUDGET_USD = 2

SAFETY_MARGIN_PERCENT = 20

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [4]:
@dataclass
class FeatureBudget:

    requests_per_day: int
    avg_input_tokens: int
    avg_output_tokens: int
    peak_requests_per_minute: int

    input_price_per_million: float
    output_price_per_million: float

    rpm_limit: int
    tpm_limit: int

    monthly_budget_usd: float
    daily_budget_usd: float

    safety_margin_percent: float

    def calculate(self):

        if min(
            self.requests_per_day,
            self.avg_input_tokens,
            self.avg_output_tokens,
            self.peak_requests_per_minute
        ) < 0:

            raise ValueError("Usage assumptions cannot be negative.")

        if self.safety_margin_percent < 0:
            raise ValueError("Safety margin cannot be negative.")

        if min(
            self.input_price_per_million,
            self.output_price_per_million,
            self.monthly_budget_usd,
            self.daily_budget_usd
        ) < 0:

            raise ValueError("Prices and budgets cannot be negative.")

        if self.rpm_limit < 0 or self.tpm_limit < 0:
            raise ValueError("Rate limits cannot be negative.")

        # Token calculations

        input_day = (
            self.requests_per_day *
            self.avg_input_tokens
        )

        output_day = (
            self.requests_per_day *
            self.avg_output_tokens
        )

        total_day = input_day + output_day

        # Cost calculations

        daily_cost = (
            input_day * self.input_price_per_million
            +
            output_day * self.output_price_per_million
        ) / 1_000_000

        monthly_cost = daily_cost * 30

        # Safety margin

        margin = 1 + self.safety_margin_percent / 100

        peak_rpm = (
            self.peak_requests_per_minute * margin
        )

        peak_tpm = (
            self.peak_requests_per_minute
            *
            (
                self.avg_input_tokens +
                self.avg_output_tokens
            )
            *
            margin
        )

        return {

            "input_tokens_day": input_day,

            "output_tokens_day": output_day,

            "total_tokens_day": total_day,

            "daily_cost_usd": daily_cost,

            "monthly_cost_usd": monthly_cost,

            "peak_rpm_with_margin": peak_rpm,

            "peak_tpm_with_margin": peak_tpm,

            "rpm_ok": peak_rpm <= self.rpm_limit,

            "tpm_ok": peak_tpm <= self.tpm_limit,

            "daily_budget_ok": (
                daily_cost * margin <= self.daily_budget_usd
            ),

            "monthly_budget_ok": (
                monthly_cost * margin <= self.monthly_budget_usd
            )
        }

In [5]:
budget = FeatureBudget(

    requests_per_day=REQUESTS_PER_DAY,

    avg_input_tokens=AVG_INPUT_TOKENS,

    avg_output_tokens=AVG_OUTPUT_TOKENS,

    peak_requests_per_minute=PEAK_REQUESTS_PER_MINUTE,

    input_price_per_million=INPUT_PRICE_PER_MILLION,

    output_price_per_million=OUTPUT_PRICE_PER_MILLION,

    rpm_limit=RPM_LIMIT,

    tpm_limit=TPM_LIMIT,

    monthly_budget_usd=MONTHLY_BUDGET_USD,

    daily_budget_usd=DAILY_BUDGET_USD,

    safety_margin_percent=SAFETY_MARGIN_PERCENT
)

result = budget.calculate()

print("Budget calculation completed!")

Budget calculation completed!


In [6]:
print("\n===================================")
print("       TOKEN ECONOMICS REPORT")
print("===================================")

print("Input tokens/day:",
      f"{result['input_tokens_day']:,}")

print("Output tokens/day:",
      f"{result['output_tokens_day']:,}")

print("Total tokens/day:",
      f"{result['total_tokens_day']:,}")

print("Estimated daily cost:",
      f"${result['daily_cost_usd']:.4f}")

print("Estimated monthly cost:",
      f"${result['monthly_cost_usd']:.2f}")

print("Peak RPM with margin:",
      f"{result['peak_rpm_with_margin']:.1f}")

print("Peak TPM with margin:",
      f"{result['peak_tpm_with_margin']:.0f}")

print("\n===================================")
print("             CHECKS")
print("===================================")

checks = {
    "RPM Capacity": result["rpm_ok"],
    "TPM Capacity": result["tpm_ok"],
    "Daily Budget": result["daily_budget_ok"],
    "Monthly Budget": result["monthly_budget_ok"]
}

for name, passed in checks.items():

    print(
        name,
        ":",
        "PASS" if passed else "REVIEW"
    )

if all(checks.values()):

    print("\nRecommendation: WITHIN configured planning limits.")

else:

    print("\nRecommendation: REVISE assumptions, limits or design.")


       TOKEN ECONOMICS REPORT
Input tokens/day: 2,400,000
Output tokens/day: 600,000
Total tokens/day: 3,000,000
Estimated daily cost: $1.4400
Estimated monthly cost: $43.20
Peak RPM with margin: 36.0
Peak TPM with margin: 54000

             CHECKS
RPM Capacity : PASS
TPM Capacity : PASS
Daily Budget : PASS
Monthly Budget : REVIEW

Recommendation: REVISE assumptions, limits or design.


In [7]:
import unittest

class TestFeatureBudget(unittest.TestCase):

    def make_budget(self, **overrides):

        values = dict(

            requests_per_day=1000,

            avg_input_tokens=1000,

            avg_output_tokens=500,

            peak_requests_per_minute=10,

            input_price_per_million=1.0,

            output_price_per_million=2.0,

            rpm_limit=100,

            tpm_limit=50000,

            monthly_budget_usd=100,

            daily_budget_usd=5,

            safety_margin_percent=20
        )

        values.update(overrides)

        return FeatureBudget(**values)

    def test_cost_estimate(self):

        result = self.make_budget().calculate()

        self.assertAlmostEqual(
            result["daily_cost_usd"], 2.0
        )

        self.assertAlmostEqual(
            result["monthly_cost_usd"], 60.0
        )

    def test_capacity_passes(self):

        result = self.make_budget().calculate()

        self.assertTrue(result["rpm_ok"])
        self.assertTrue(result["tpm_ok"])

    def test_rpm_limit_fails(self):

        result = self.make_budget(
            rpm_limit=5
        ).calculate()

        self.assertFalse(result["rpm_ok"])

    def test_monthly_budget_fails(self):

        result = self.make_budget(
            monthly_budget_usd=20
        ).calculate()

        self.assertFalse(result["monthly_budget_ok"])


suite = unittest.defaultTestLoader.loadTestsFromTestCase(
    TestFeatureBudget
)

unittest.TextTestRunner(
    verbosity=2
).run(suite)

test_capacity_passes (__main__.TestFeatureBudget.test_capacity_passes) ... ok
test_cost_estimate (__main__.TestFeatureBudget.test_cost_estimate) ... ok
test_monthly_budget_fails (__main__.TestFeatureBudget.test_monthly_budget_fails) ... ok
test_rpm_limit_fails (__main__.TestFeatureBudget.test_rpm_limit_fails) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

In [9]:
import pandas as pd

In [10]:
scenarios = {

    "Original": {
        "requests_per_day": 2000,
        "avg_input_tokens": 1200,
        "avg_output_tokens": 300
    },

    "More Users": {
        "requests_per_day": 4000,
        "avg_input_tokens": 1200,
        "avg_output_tokens": 300
    },

    "Longer Context": {
        "requests_per_day": 2000,
        "avg_input_tokens": 2000,
        "avg_output_tokens": 300
    },

    "Longer Answers": {
        "requests_per_day": 2000,
        "avg_input_tokens": 1200,
        "avg_output_tokens": 600
    },

    "Reduced Prompt": {
        "requests_per_day": 2000,
        "avg_input_tokens": 600,
        "avg_output_tokens": 300
    }
}

scenario_results = []

for name, values in scenarios.items():

    test_budget = FeatureBudget(

        **values,

        peak_requests_per_minute=30,

        input_price_per_million=0.30,

        output_price_per_million=1.20,

        rpm_limit=60,

        tpm_limit=100000,

        monthly_budget_usd=50,

        daily_budget_usd=2,

        safety_margin_percent=20
    )

    output = test_budget.calculate()

    scenario_results.append({

        "Scenario": name,

        "Input Tokens/Day": output["input_tokens_day"],

        "Output Tokens/Day": output["output_tokens_day"],

        "Daily Cost ($)": round(
            output["daily_cost_usd"], 4
        ),

        "Monthly Cost ($)": round(
            output["monthly_cost_usd"], 2
        ),

        "RPM Check": output["rpm_ok"],

        "TPM Check": output["tpm_ok"]
    })


scenario_df = pd.DataFrame(scenario_results)

display(scenario_df)

,Scenario,Input Tokens/Day,Output Tokens/Day,Daily Cost ($),Monthly Cost ($),RPM Check,TPM Check
0,Original,2400000,600000,1.44,43.2,True,True
1,More Users,4800000,1200000,2.88,86.4,True,True
2,Longer Context,4000000,600000,1.92,57.6,True,True
3,Longer Answers,2400000,1200000,2.16,64.8,True,True
4,Reduced Prompt,1200000,600000,1.08,32.4,True,True
